# JEI 同期基準重播
在已驗證的 JEI main checkout 根目錄執行。將同批 benchmark-sources.json 放在根目錄。產物僅供研究，不能用於合格 OOS 或實盤。

In [ ]:
import json
from collections import Counter
from pathlib import Path
from scripts.collect_pick_snapshots import load_snapshots
from scripts.corporate_actions import actions_between, validated_actions
from scripts.forward_pick_score import COHORTS, HORIZONS
from scripts.paper_benchmark import compare_with_benchmark
from scripts.paper_evidence import wilson_interval
from scripts.paper_report import build_report
from scripts.paper_score_summary import summarize
from scripts.pick_risk_metrics import max_close_drawdown

source = json.loads(Path("benchmark-sources.json").read_text())
series = {name: block["closes"] for name, block in source["series"].items()}
calendar = [row["date"] for row in series["TAIEX"]]
assert calendar == [row["date"] for row in series["0050"]]
assert calendar[-1] == "2026-11-11"
archive = json.loads(Path("remote/daily_prices.json").read_text())
events = validated_actions(archive)
picks = load_snapshots(Path("data/pick_snapshots"))
assert len(picks) == 341 and max(p["as_of"] for p in picks) == calendar[-1]
prices = {code: [{"date": day, "close": archive["days"][day][code]} for day in calendar
                 if code in archive["days"].get(day, {})]
          for code in sorted({p["code"] for p in picks})}
close_by_symbol = {code: {row["date"]: row["close"] for row in rows}
                   for code, rows in prices.items()}
naive = build_report(picks, prices)["picks"]
dated = build_report(picks, prices, series["TAIEX"])["picks"]
assert len(dated) == len(naive)

# Apply known corporate action exclusions before benchmarking or counting wins.
for pick in dated:
    for outcome in pick["outcomes"].values():
        if outcome["status"] != "mature":
            continue
        start, end = pick["entry_date"], outcome["exit_date"]
        if actions_between(events, pick["code"], start, end):
            outcome.update(status="unavailable", reason="known_unadjusted_corporate_action")
            continue
        symbol_closes = close_by_symbol[pick["code"]]
        path = [symbol_closes.get(day) for day in calendar if start < day <= end]
        if None in path:
            outcome.update(status="unavailable", reason="missing_intermediate_close")
            continue
        outcome["close_drawdown_pct"] = max_close_drawdown(pick["entry_price"], path)

comparisons = {}
for name, rows in series.items():
    benchmark = {row["date"]: row["close"] for row in rows}
    windows = {
        (pick["entry_date"], outcome["exit_date"]):
            (benchmark[outcome["exit_date"]] / benchmark[pick["entry_date"]] - 1) * 100
        for pick in dated for outcome in pick["outcomes"].values()
        if outcome["status"] == "mature"
    }
    comparisons[name] = compare_with_benchmark(dated, windows)

summary = summarize(dated)
for cohort in COHORTS:
    for horizon in map(str, HORIZONS):
        outcomes = [p["outcomes"][horizon] for p in dated if p["cohort"] == cohort]
        finished = [o for o in outcomes if o["status"] == "mature"]
        summary[cohort][horizon].update(
            counts=dict(Counter(o.get("reason", o["status"]) for o in outcomes)),
            win_rate_ci95_pct=wilson_interval(sum(o["net_return_pct"] > 0 for o in finished), len(finished)),
            worst_trade_close_drawdown_pct=min((o["close_drawdown_pct"] for o in finished), default=None),
        )

counts = {}
for h in map(str, HORIZONS):
    counts[h] = {
        "symbol_dates_only_matured": sum(p["outcomes"][h]["status"] == "mature" for p in naive),
        "exchange_calendar_matured": sum(p["outcomes"][h]["status"] == "mature" for p in dated),
        "matured_by_different_exit": sum(
            p["outcomes"][h]["status"] == q["outcomes"][h]["status"] == "mature"
            and p["outcomes"][h]["exit_date"] != q["outcomes"][h]["exit_date"]
            for p, q in zip(naive, dated)
        ),
    }

result = {
    "kind": "diagnostic_price_only_same_window_benchmarks",
    "signal_range": [min(p["as_of"] for p in picks), max(p["as_of"] for p in picks)],
    "signals": len(picks),
    "distinct_date_symbol_signals": len({(p["as_of"], p["code"]) for p in picks}),
    "benchmark_sessions": len(calendar),
    "missing_archive_sessions_since_first_signal": [day for day in calendar if day >= min(p["as_of"] for p in picks) and day not in archive["days"]],
    "method": "frozen snapshots; next-session close proxy; 3/7/15/30/60 sessions from entry",
    "strategy_version": "diagnostic existing forward_pick_score defaults; mixed historical JEI signal versions",
    "strategy_costs": {"buy_fee": 0.001425, "sell_fee": 0.001425, "sell_tax": 0.003, "slippage_per_side": 0.001},
    "benchmark_costs": "gross price changes; dividends and fund trading costs excluded",
    "limitations": "Not executable fills or walk-forward. Corporate actions checked only where archived. Wilson intervals descriptive, observations overlap. Drawdown is individual trade path, not portfolio drawdown.",
    "qualified_oos": False,
    "comparison": comparisons,
    "summary": summary,
    "calendar_diagnostic": counts,
}
Path("benchmark-result.json").write_text(json.dumps(result, ensure_ascii=False, sort_keys=True, indent=2) + "\n")
print(json.dumps({"signals": result["signals"], "missing_archive_sessions": result["missing_archive_sessions_since_first_signal"],
                  "calendar_diagnostic": counts}, ensure_ascii=False))
